# Illustrating a book using Nano Banana and genMedia models

<a target="_blank" href="https://colab.research.google.com/github/google-gemini/cookbook/blob/main/examples/Book_illustration.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" height=30/></a>

In this guide, you are going to use multiple Gemini features (long context, multimodality, structured output, file API, chat mode...) in conjunction with the [Nano Banana](../quickstarts/Get_Started_Nano_Banana.ipynb) image generation model to illustrate a book.

You will also explore how to bring your illustrations to life with:
- 🎬 **[Veo](../quickstarts/Get_started_Veo.ipynb)** — Animate a chapter illustration into a short video
- 🎵 **[Lyria](../quickstarts/Get_started_Lyria.ipynb)** — Generate instrumental background music for each chapter
- 🗣️ **[TTS](../quickstarts/Get_started_TTS.ipynb)** — Have a narrator read the opening of a chapter aloud

Each concept will be explained along the way, but if you need a simpler introduction to Gemini Image generation model, check the [getting started](../quickstarts/Get_Started_Nano_Banana.ipynb) notebook, or the [Image generation documentation](https://ai.google.dev/gemini-api/docs/image-generation).

Note: for the sake of the notebook's size (and your billing if you run it), the number of images has been limited to 3 characters and 3 chapters each time, but feel free to remove the limitation if you want more with your own experimentations.

Also note that this notebook used to use [Imagen](https://ai.google.dev/gemini-api/docs/imagen) models instead of Nano Banana. If you are interested in the Imagen version, checked-out this [old version](../../c604f672f621186f609b1d977a918250eaca19f2/examples/Book_illustration.ipynb).

> **Note:** [Enable billing](https://ai.google.dev/gemini-api/docs/billing#enable-cloud-billing) to use Image Generation. This is a pay-as-you-go feature (cf. [pricing](https://ai.google.dev/pricing#gemini-2.5-flash-image-preview)). This does **not** apply if you use `gemini-2.5-flash-image` (Nano Banana) which has a free tier.
>
> This notebook also includes optional sections for **Video generation** ([Veo](../quickstarts/Get_started_Veo.ipynb)), **Music generation** ([Lyria](../quickstarts/Get_started_Lyria.ipynb)), and **Text-to-Speech** ([TTS](../quickstarts/Get_started_TTS.ipynb)) which are all paid features. Each has its own opt-in checkbox that you need to enable before running.

## 0/ Setup

This section install the SDK, set it up using your [API key](../quickstarts/Authentication.ipynb), imports the relevant libs, downloads the sample videos and upload them to Gemini.

Just collapse (click on the little arrow on the left of the title) and run this section if you want to jump straight to the examples (just don't forget to run it otherwise nothing will work).

### Install SDK


In [1]:
%pip install -U -q "google-genai>=2.10.0" # 2.10 for interactions API

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 16.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 259.1/259.1 kB 13.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.49.0, but you have google-auth 2.56.3 which is incompatible.


### Setup your API key

To run the following cell, your API key must be stored it in a Colab Secret named `GEMINI_API_KEY`. If you don't already have an API key, or you're not sure how to create a Colab Secret, see [Authentication ![image](https://storage.googleapis.com/generativeai-downloads/images/colab_icon16.png)](../quickstarts/Authentication.ipynb) for an example.

In [4]:
from google.colab import userdata

GEMINI_API_KEY="YOUR_GEMINI_API_KEY"

### Initialize SDK client

With the new SDK you now only need to initialize a client with you API key (or OAuth if using [Vertex AI](https://link_to_vertex_AI)). The model is now set in each call.

In [5]:
from google import genai
from google.genai import types

client = genai.Client(
    api_key=GEMINI_API_KEY,
    http_options=types.HttpOptions(
        retry_options=types.HttpRetryOptions(
            attempts=5,
            initial_delay=2.0,
            max_delay=60.0,
            http_status_codes=[429, 500, 502, 503, 504]
        )
    )
)

### Imports

Some imports to display markdown text and images in Colab.

In [6]:
import json
from PIL import Image
from IPython.display import display, Markdown

### Select models

Select the models you qre going to use and confirm that you are aware that some of those models don't have a free tier, so running the notebook might cost you a bit.

You can also use the [`priority` service tier](https://ai.google.dev/gemini-api/docs/priority-inference) to be certain your requests will go through (but be careful, it means they will be twice as expensive).

In [7]:
IMAGE_MODEL_ID = "gemini-3.1-flash-lite-image"  # @param ["gemini-3.1-flash-lite-image", "gemini-2.5-flash-image", "gemini-3.1-flash-image-preview", "gemini-3-pro-image-preview"] {"allow-input":true, isTemplate: true}
GEMINI_MODEL_ID = "gemini-3.6-flash" # @param ["gemini-2.5-flash", "gemini-3.6-flash", "gemini-3.1-pro-preview"] {"allow-input":true, isTemplate: true}

service_tier = "flex" # @param ["flex","standard","priority"]

For the sake of the notebook's size (and your billing if you run it), the number of images has been limited to 5 characters and 3 chapters each time, but feel free to remove the limitation if you want more with your own experimentations.

In [8]:
max_character_images = 5 # @param {type:"integer",isTemplate: true, min:1}
max_chapter_images = 3 # @param {type:"integer",isTemplate: true, min:1}

# Illustrate a book: The Wind in the Willows

## 1/ Get a book and upload using the File API

Start by downloading a book from the open-source [Project Gutenberg](www.gutenberg.org) library. For example, it can be [The Wind in the Willows](https://en.wikipedia.org/wiki/The_Wind_in_the_Willows) from Kenneth Grahame.

The file API (`client.files.upload`) are used to upload the file so that Gemini can easily access it.

In [9]:
import requests

url = "https://www.gutenberg.org/cache/epub/289/pg289.txt"  # @param {type:"string"}

response = requests.get(url)
with open("book.txt", "wb") as file:
    file.write(response.content)

book = client.files.upload(file="book.txt")


Let's also define some more instructions which will act as "system instructions" or a negative prompt to tell the model what you do not want to see (text on the images).

In [10]:
system_instructions = """
  There must be no text on the image, it should not look like a cover page.
  It should be an full illustration with no borders, titles, nor description.
  Unless asked otherwise, stay family-friendly with uplifting colors.
  Each produced should be a simple image, no panels.
"""

## 2/ Start the chat

You are going to chain interactions via their IDs here so that Gemini will keep the history of what you asked it, and also so that you don't have to send it the book every time. More details on chat mode in the [Get Started](https://colab.sandbox.google.com/github/google-gemini/cookbook/blob/main/quickstarts/Get_started.ipynb#chat) notebook.

You should also define the format of the output you want using [structured output](https://ai.google.dev/gemini-api/docs/structured-output?lang=python#generate-json). You will mainly use Gemini to generate prompts so let's define a Pydantic model with two fields, a name and a prompt:

In [11]:
from pydantic import BaseModel

class Prompt(BaseModel):
    name: str
    prompt: str


`client.interactions.create` starts the chat and defines its main parameters (model and the output you want).

In [12]:
# Start the conversation with the book content
book_interaction = client.interactions.create(
    model=GEMINI_MODEL_ID,
    input=[
        {"type": "text", "text": "Here's a book, to illustrate using Nano Banana. Don't say anything for now, instructions will follow."},
        {"type": "document", "uri": book.uri},
    ],
    service_tier=service_tier,
)


The first message sent to the model is just to give it a bit of context ("*to illustrate using Nano Banana*"), and more importantly give it the book.

It could have been done during the next step, especially since you're not interested in what the model has to say this time, but splitting the two steps makes it clearer.

## 3/ Define a style

If you want to test a specific style, just write it down and Gemini will use it. Still, tell Gemini about it so it will adapt the prompts it will generate accordingly.

If you prefer to let Gemini choose the best style for the book, leave the style empty and ask Gemini to define a style fitting to the book.

In [13]:
style = "" # @param {type:"string", "placeholder":"Write your own style or leave empty to let Gemini generate one"}

if style=="":
  style_interaction = client.interactions.create(
      model=GEMINI_MODEL_ID,
      input="Can you define a art style that would fit the story but with a twist? Just give us the prompt for the art syle that will added to the furture prompts.",
      previous_interaction_id=book_interaction.id,
      service_tier=service_tier,
  )
  last_interaction = style_interaction
  style = style_interaction.output_text
else:
  style_interaction = client.interactions.create(
      model=GEMINI_MODEL_ID,
      input=f'The art style will be:"{style}". Keep that in mind when generating future prompts. Keep quiet for now, instructions will follow.',
      previous_interaction_id=book_interaction.id,
      service_tier=service_tier,
  )
  last_interaction = style_interaction

display(Markdown(f"### Style:"))
print(style)

style = f'Follow this style: "{style}" '


### Style:

Here is an art style definition with a **"Dark Autumnal Fairytale & Needle-Felted Diorama"** twist—blending classic Beatrix Potter/Arthur Rackham vintage illustration with hyper-tactile stop-motion aesthetics (think *Over the Garden Wall* meets *Fantastic Mr. Fox*).

---

### **Style Prompt Block:**

> **Art Style:** Soft needle-felted wool and hand-carved miniature diorama style, cozy British Edwardian pastoral gothic, intricate Beatrix Potter character designs with a moody cinematic twist, warm amber and moss-green color palette, misty vintage film lighting, detailed mossy textures, weathered tweed and velvet garments, soft volumetric golden hour sunlight, nostalgic 19th-century storybook illustration aesthetic, cinematic depth of field, 35mm film grain --ar 16:9 --v 6.0


## 4/ Generate portraits of the main characters

You are now ready to start generating images, starting with the main characters.

Ask Gemini to describe each of the main characters (excluding children as Nano Banana can't generate images of them in EEA) and check that the output follows the format requested.


In [14]:
characters_prompts_interaction = client.interactions.create(
    model=GEMINI_MODEL_ID,
    input="Can you describe the main characters (only the adults) and prepare a prompt describing them with as much details as possible (use the descriptions from the book) so Nano Banana can generate images of them? Each prompt should be at least 50 words.",
    previous_interaction_id=style_interaction.id,
    response_format={
        "type": "text",
        "mime_type": "application/json",
        "schema": {"type": "array", "items": Prompt.model_json_schema()},
    },
    service_tier=service_tier,
)
last_interaction = characters_prompts_interaction

characters = json.loads(characters_prompts_interaction.output_text)

print(json.dumps(characters, indent=4))


[
    {
        "name": "Mole",
        "prompt": "A detailed full-body character portrait of Mole, a gentle anthropomorphic European mole featuring soft plush black fur, small round ears, sensitive whiskers, and neat pink paws. He is elegantly dressed in a rich black velvet smoking-suit with smooth lapels and warm trousers. He stands inside a cozy, subterranean hallway illuminated by soft candlelight, looking earnest and curious."
    },
    {
        "name": "Water Rat",
        "prompt": "A detailed full-body character portrait of Water Rat, a suave anthropomorphic water vole with a grave brown face, sleek whiskers, and thick silky brown fur. He wears a tailored river-green tweed waistcoat, a neatly tied cravat, and a crisp white collar. He stands proudly on a grassy riverbank holding a wooden oar, with calm river waters and reeds in the background."
    },
    {
        "name": "Mr. Badger",
        "prompt": "A detailed full-body character portrait of Mr. Badger, a tall and broad 

Now that you have the prompts, you just need to loop on all the characters and have Nano Banana generate an image for them. This model uses the same API as the text generation models.

Like before, for the sake of consistency, we are going to use chat mode, but within a different instance.

For an extensive explanation on the Nano Banana model and its options, check the [getting started with Nano Banana](../quickstarts/Get_Started_Nano_Banana.ipynb) notebook. But here's a quick overview of what being used here:
* `prompt` is the prompt passed down to Nano Banana. You're not just sending what Gemini has generate to describe the chacaters but also our style and our system instructions.
* `response_modalities=['Image']` because we only want images
* `aspect_ratio="9:16"` because we want portraits images

Note that we could have used system instructions but the model currently ignores them so we decided to pass them as message.

In [15]:
# TODO: try using the last interaction (characters_prompts_interaction)

In [16]:
character_images = []
last_image_interaction = None

# Set up the image generation context
# TODO: do we need the first turn?
characters_image_interaction = client.interactions.create(
    model=IMAGE_MODEL_ID,
    input=f"""
      You are going to generate portrait images to illustrate The Wind in the Willows from Kenneth Grahame.
      The style we want you to follow is: {style}
      Also follow those rules: {system_instructions} # TODO: Sysyem instructions
    """,
    service_tier=service_tier,
)

for character in characters[:max_character_images]:
  display(Markdown(f"### {character['name']}"))
  display(Markdown(character['prompt']))

  characters_image_interaction = client.interactions.create(
      model=IMAGE_MODEL_ID,
      input=f"Create an illustration for {character['name']} following this description: {character['prompt']}",
      previous_interaction_id=characters_image_interaction.id,
      service_tier=service_tier,
  )

  # Extract image from interaction steps
  # TODO: try output_image
  generated_image = None
  for step in reversed(characters_image_interaction.steps):
      if step.type == "model_output" and step.content:
          for content in reversed(step.content):
              if content.type == "image":
                  generated_image = content
                  break
          if generated_image:
              break

  if generated_image:
      from IPython.display import display as disp, HTML
      import base64
      img_html = f'<img src="data:{generated_image.mime_type};base64,{generated_image.data}" style="max-width:512px" />'
      disp(HTML(img_html))
  else:
      print(f"No image generated for {character['name']}")

  character_images.append(generated_image)

last_image_interaction = characters_image_interaction
# Be careful; long output (see below)


RateLimitError: Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_input_token_count, limit: 0, model: gemini-3.1-flash-lite-image\n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-3.1-flash-lite-image\n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-3.1-flash-lite-image\nPlease retry in 35.136980262s.', 'code': 'too_many_requests'}}

## 5/ Illustrate the chapters of the book

After the characters, it's now time to create illustrations for the content of the book. You are going to ask Gemini to generate prompts for each chapter and then ask Nano Banana to generate images based on those prompts.

In [ ]:
chapters_prompts_interaction = client.interactions.create(
    model=GEMINI_MODEL_ID,
    input="Now, for each chapters of the book, give me a prompt to illustrate what happens in it. It should be a single image, not a multi-tiled page. Be very descriptive, especially of the characters. Be very descriptive and remember to tell their name and to reuse the character prompts if they appear in the images. Also list all characters who appear in it.",
    previous_interaction_id=characters_prompts_interaction.id,
    response_format={
        "type": "text",
        "mime_type": "application/json",
        "schema": {"type": "array", "items": Prompt.model_json_schema()},
    },
    service_tier=service_tier,
)
last_interaction = chapters_prompts_interaction

chapters = json.loads(chapters_prompts_interaction.steps[-1].content[0].text)[:max_chapter_images]

print(json.dumps(chapters, indent=4))


In [ ]:
chapters_image_interaction = client.interactions.create(
    model=IMAGE_MODEL_ID,
    input="Starting from now, we're going to illustrate the book's chapters. Don't forget to refer to your previous illustrations of the characters to keep the characters consistency, but feel free to change their position.",
    previous_interaction_id=last_image_interaction.id,
    service_tier=service_tier,
)
last_image_interaction = chapters_image_interaction

for chapter in chapters:
  display(Markdown(f"### {chapter['name']}"))
  display(Markdown(chapter['prompt']))

  chapters_image_interaction = client.interactions.create(
      model=IMAGE_MODEL_ID,
      input=f"Create an illustration for {chapter['name']} using the previously generated characters following this description: {chapter['prompt']}",
      previous_interaction_id=last_image_interaction.id,
      service_tier=service_tier,
  )
  last_image_interaction = chapters_image_interaction

  # TODO: use output_image?
  for step in reversed(chapters_image_interaction.steps):
      if step.type == "model_output" and step.content:
          for content in reversed(step.content):
              if content.type == "image":
                  generated_image = content
                  from PIL import Image as PILImage
                  import io, base64
                  img = PILImage.open(io.BytesIO(base64.b64decode(content.data)))
                  display(img)
                  break
          break

# Be careful; long output (see below)


### Bonus: Going further with more granular control

If you have a lot of characters, want to make sure the model is using the right references, you can ask it to list all characters present in each chapter image so that you can pass only those characters's images to the model. This would also work if you have reccuring locations, items, etc...

In [ ]:
class Chapter(BaseModel):
    name: str
    prompt: str
    characters: list[str]

In [ ]:
chapters_prompts_interaction = client.interactions.create(
    model=GEMINI_MODEL_ID,
    input="Now, for each chapters of the book, give me a prompt to illustrate what happens in it. Be very descriptive, especially of the characters. Be very descriptive and remember to tell their name and to reuse the character prompts if they appear in the images. Also list all characters who appear in it.",
    previous_interaction_id=characters_prompts_interaction.id,
    response_format={
        "type": "text",
        "mime_type": "application/json",
        "schema": {"type": "array", "items": Chapter.model_json_schema()},
    },
    service_tier=service_tier,
)
last_interaction = chapters_prompts_interaction

chapters = json.loads(chapters_prompts_interaction.steps[-1].content[0].text)[:max_chapter_images]

print(json.dumps(chapters, indent=4))


In [ ]:
# @title Character ref image finder
import base64
from typing import List
from google.genai import types

def get_character_references_images(
    requested_character_names: List[str]
) -> List:
    """
    Takes a list of character names and returns a flattened list of
    character images ready to be fed into Gemini's send_message as content parts,
    using the global 'characters' and 'character_images' variables.

    Args:
        requested_character_names: A list of strings, where each string is a character's name.

    Returns:
        A list of image objects for the requested characters.
    """
    gemini_content_parts = []

    # Create a mapping from character name to its data and image for efficient lookup
    character_map = {}
    for i, char_data in enumerate(characters):
        if i < len(character_images): # Ensure there's a corresponding image
            character_map[char_data['name']] = (char_data['prompt'], character_images[i])
        else:
            print(f"Warning: No image found for character '{char_data['name']}' at index {i}. Skipping image for this character.")
            character_map[char_data['name']] = (char_data['prompt'], None) # Or handle as needed

    for char_name in requested_character_names:
        if char_name in character_map:
            char_prompt, char_image = character_map[char_name]

            if char_image:
                # Decode the base64 string to bytes
                image_bytes = base64.b64decode(char_image.data)
                gemini_content_parts.append(types.Part.from_bytes(data=image_bytes, mime_type=char_image.mime_type))
            else:
                print(f"Warning: No image available for {char_name}.")
        else:
            print(f"Warning: Character '{char_name}' not found in the available characters.")

    return gemini_content_parts